# RF-DETR → ExecuTorch Export

Export an RF-DETR detector to a portable **ExecuTorch program** (`.pte`) for on-device deployment on
mobile and edge hardware. Unlike ONNX or TFLite, the model is captured directly via `torch.export`
(no intermediate conversion) and lowered to a hardware backend:

| Backend | Target | Precision |
|---------|--------|-----------|
| **`xnnpack`** (this notebook) | portable CPU (Android / iOS / Linux / macOS) | fp32 |
| `coreml` | Apple Neural Engine | fp16 |
| `qnn` | Qualcomm Snapdragon HTP | fp16 |

!!! warning "On-device inference is currently blocked by an ExecuTorch bug"

    This cookbook covers **export only**. On `executorch==1.3.1` (the latest release), running the exported
    `.pte` returns **incorrect detections on real images** — the `to_edge` lowering corrupts the DINOv2
    backbone output, and this reproduces on **both XNNPACK and CoreML**. `torch.export` itself is bit-exact
    with eager PyTorch, so the model export is correct; the fault is in ExecuTorch's ahead-of-time lowering.
    Track the fix in the linked issue before relying on `.pte` inference.

## 1. Install

The `[executorch]` extra provides the exporter and the XNNPACK backend.

Colab ships mutually inconsistent preinstalled packages that otherwise crash `import rfdetr`, so two are
aligned: `torchaudio` is uninstalled (RF-DETR never uses it, but `transformers` imports it when present and
a `torch`/`torchaudio` CUDA-version mismatch then errors), and `pillow` is force-reinstalled to a clean
version (a half-upgraded PIL breaks `torchvision`'s import with `cannot import name '_Ink'`).

> **`flatc`** — ExecuTorch serializes the `.pte` with the FlatBuffers compiler. It ships in the Linux
> `executorch` wheel (so Colab works out of the box); on macOS install it separately with
> `brew install flatbuffers`.

> **Colab**: after this cell, **Runtime → Restart session**, then run from the next cell — Colab keeps the
> old package versions loaded until a restart.

In [ ]:
!pip install -q "rfdetr[executorch]>=1.9.0" "torch<2.13" supervision
!pip install -q --force-reinstall --no-deps "pillow==11.3.0"
!pip uninstall -q -y torchaudio

## 2. Setup

ExecuTorch export is CPU-only — no GPU is needed.

In [ ]:
from pathlib import Path

from rfdetr import RFDETRSmall

EXPORT_DIR = Path("export_executorch")
EXPORT_DIR.mkdir(exist_ok=True)

## 3. Export to a `.pte` program

`format="executorch"` requires an explicit `backend`. The COCO-pretrained `RFDETRSmall` is exported
directly — pass `pretrain_weights="<path/to/checkpoint.pth>"` to export a fine-tuned model instead.

The file is named after the model variant (`rfdetr-small.pte`). `torch.export` bakes a fixed input shape
into the graph (square at the model's resolution), so `dynamic_batch` is not supported — export one `.pte`
per batch size.

In [ ]:
model = RFDETRSmall()

pte_path = model.export(format="executorch", backend="xnnpack", output_dir=str(EXPORT_DIR))
print(f"ExecuTorch program: {pte_path}  ({pte_path.stat().st_size / 1e6:.1f} MB)")

## 4. Other backends

Swap the `backend` argument to target a different on-device runtime:

=== "Apple (CoreML, fp16)"

    ```python
    # requires: pip install coremltools
    model.export(format="executorch", backend="coreml", output_dir="export_executorch")
    ```

=== "Qualcomm Snapdragon (QNN, fp16)"

    ```python
    # requires an ExecuTorch source build against the QAIRT SDK (not pip-installable)
    model.export(format="executorch", backend="qnn", soc="SM8650", output_dir="export_executorch")
    ```

CoreML runs fp16 on the Apple Neural Engine; QNN targets the Snapdragon HTP and bakes in the target SoC.

## 5. Running the exported `.pte` (reference)

!!! warning "Known bug — returns 0 detections on real images (executorch 1.3.1)"

    The code below is the **intended** inference path. On the current ExecuTorch release it produces
    incorrect detections (see the top-of-notebook warning). It is kept here as a reference for when the
    upstream lowering bug is fixed. Validate against eager PyTorch on your target before relying on it.

The `.pte` expects the same input the model was trained on: NCHW, ImageNet-normalized, at the traced
resolution. `Runtime.load_program(...).load_method("forward")` gives a callable graph whose two outputs
are `dets` (boxes, `cxcywh`, normalized) and `labels` (class logits).

```python
import torch
from executorch.runtime import Runtime
from PIL import Image
from rfdetr.export.benchmark import infer_transforms, post_process

image = Image.open("image.jpg").convert("RGB")
resolution = model.model_config.resolution
transforms = infer_transforms((resolution, resolution))
image_tensor, _ = transforms(image, None)

method = Runtime.get().load_program(str(pte_path)).load_method("forward")
dets, labels = method.execute([image_tensor[None].float()])

target_sizes = torch.tensor([[image.height, image.width]])
result = post_process({"dets": dets, "labels": labels}, target_sizes)[0]
# result["boxes"], result["scores"], result["labels"] — filter by confidence, then annotate.
```

## Next steps

- **Deploy on-device** — copy the `.pte` to your Android / iOS / edge app and run it with the ExecuTorch
  runtime for that platform. See the [ExecuTorch docs](https://pytorch.org/executorch/).
- **Fine-tuned weights** — pass `pretrain_weights="<path/to/checkpoint.pth>"` when constructing the model.
- **For working inference today**, use the [TensorRT cookbook](export-tensorrt/) (NVIDIA GPU) or
  [`inference-models`](https://github.com/roboflow/inference/tree/main/inference_models) (PyTorch / ONNX / TensorRT).
- See the [Export documentation](https://rfdetr.roboflow.com/learn/export/) for all formats and options.